# 02 · Constraints: distances, angles, dihedrals, planes

`constrain=` sets a soft window; `fix=` locks geometry exactly. Both take distances, angles, and
dihedrals (0-based atom-index keys); `fix=` also takes coordinates or atom lists. `.measure(key)`
reads a result back.

In [ ]:
import numpy as np
import rxembed as rx
import xyzrender
from rdkit import Chem

rx.set_verbose("INFO")

## A distance window (by index)

Set a target with `constrain={(i, j): value}`. rxembed makes a small distance window for the embed and restrained UFF relax. Pass `(lo, hi)` only when the window itself matters.

In [ ]:
ACID = "O=C(O)CCCCc1ccccc1"
target = 2.8
ens = rx.embed(ACID, constrain={(1, 9): target}).mc().minimize().prune()
ens.filter("geometry")
print("d(1,9) realised:", {k: round(v, 2) for k, v in ens.measure((1, 9)).items()}, f"(target {target})")

## The same contact by SMARTS: resolve the indices yourself

Resolve a SMARTS to atom indices before constructing `constrain=`. Here a carboxylic O–H reaches a pyridine
N across two fragments. `measure()` may take SMARTS because it only queries an existing ensemble.

In [ ]:
smi = "CC(=O)O.c1ccncc1"
mol = rx.parse_smiles(smi)
(oh,), (n,) = rx.match(mol, "[OX2H]"), rx.match(mol, "n")
ens = rx.embed(smi, constrain={(oh, n): 2.8}).mc().minimize().prune()
ens.filter("geometry")
print("d(OH..n) realised:", {k: round(v, 2) for k, v in ens.measure((oh, n)).items()})

## An angle

A 3-tuple key, `constrain={(i, j, k): value}`, a single value becomes a small window (law-of-cosines leg in the bounds matrix).

In [ ]:
ens = rx.embed(ACID, constrain={(2, 1, 0): 120}).mc().minimize().prune()
ens.filter("geometry")
print("angle(2,1,0) realised:", {k: round(v, 1) for k, v in ens.measure((2, 1, 0)).items()}, "(target 120)")

## A dihedral

A 4-tuple key states a periodic dihedral. Scalar `fix` values are retained within 0.005 degrees; `constrain` is the softer window. A window such as `(170, 190)` crosses the periodic boundary, where 190 degrees is -170 degrees.

In [ ]:
torsion = (0, 1, 2, 3)
held = rx.embed("CCCC", fix={torsion: 60.0}, n=4).minimize()
soft = rx.embed("CCCC", constrain={torsion: (170.0, 190.0)}, n=4).minimize()
print("fixed 60 deg ->", {k: round(v, 3) for k, v in held.measure(torsion).items()})
print("soft 170-190 deg ->", {k: round(v, 1) for k, v in soft.measure(torsion).items()})

## Planes: a π-stack (the parallel-plane constraint)

`constrain={(ring_a, ring_b): separation}` holds two rings parallel at a target separation, atom-for-atom
(pass ring atoms in matching order, e.g. from a SMARTS match). Two phenyls of 1,4-diphenylbutane stacked at 3.7 Å.

In [ ]:
smi = "c1ccccc1CCCCc1ccccc1"
mol = Chem.AddHs(Chem.MolFromSmiles(smi))
ra, rb = [tuple(m) for m in mol.GetSubstructMatches(Chem.MolFromSmarts("c1ccccc1"))]  # the two phenyl rings
ens = rx.embed(smi, constrain={(ra, rb): 3.7}).mc(preset="ensemble").minimize().prune()
seps = [
    float(
        np.linalg.norm(
            ens.mol.GetConformer(c).GetPositions()[list(ra)].mean(0)
            - ens.mol.GetConformer(c).GetPositions()[list(rb)].mean(0)
        )
    )
    for c in ens.ids
]
print(f"pi-stack centroid separation: {min(seps):.2f}-{max(seps):.2f} A  ({len(ens)} confs, target 3.7)")
for report in ens.check().values():
    print(report.summary())  # every rendered stack is planar and clash-free
xyzrender.render(ens.mol, no_hy=True)  # the pi-contact drawn

## Incremental relaxing, and fail-loud

A tight (over-constrained) core still embeds: the triangle smoothing escalates tolerance until the bounds are metric-feasible. A genuinely impossible set (here a C–C bond stretched to 5 Å) fails loud rather than silently embedding a broken geometry.

In [ ]:
tight = rx.embed(ACID, constrain={(1, 9): (2.70, 2.75)}).mc(preset="rapid").minimize().prune()
tight.filter("geometry")
print("tight window still embeds:", len(tight), "confs; d(1,9) =", round(tight.measure((1, 9))["mean"], 2))
try:
    rx.embed("CCO", constrain={(0, 1): (5.0, 5.5)}).mc()  # a C-C bond forced to 5 A - impossible
except (ValueError, RuntimeError) as e:
    print("impossible constraint fails loud:", type(e).__name__, "-", str(e)[:55])

## The gate on a constrained periphery

The geometry gate checks that the seeded contact did not distort the rest of the molecule.

In [ ]:
ens = rx.embed(ACID, constrain={(1, 9): 2.8}).mc().minimize()
total = ens.n
ens.filter("geometry")
print(f"geometry check: {ens.n}/{total} clean")